# Assesment Notebook 

In [ ]:
# Numerical arrays
import numpy as np

#Math
import math

#C-like structs
import struct

#String numpy data type
from numpy.dtypes import StringDType

## Problem 0: GitHub Issues

This problem involves creation of Github Issues for the problems below. The issues track the progress done when solving the problems.

## Problem 1: Representing SHA-256 Data
This problem investigates how input, output and intermediate data can be represented in Python.

numpy.uint32 type is a data type in numpy that stores exactly one such number in 32 bits(32-bit unsigned integer). These are made up of 8 hex digits. Each hex digit has 4 bits(nibble).

SHA 256 implemetation flow diagram
1. Input Message - Text encoded to bytes.(Input)
2. Padding/preprocessing - Blocks of 512 bits(64 Bytes)- Intermediate
3. SHA 256 Processing- array of 16 unit32(Intermediate)
4. Compression Rounds.(Intermediate)
5. Final Hash state(Output)

![FLOWCHART](images/SHA-256%20flowchart.png)



###  Sub-section 1: 32-bit words Representation
A word is a bit string of a fixed length, called w, with 32 bits. A 32-bit word is represented as a number between 0 and 4,294,967,295 (that's 2³² − 1).
A word is a w-bit string that may be represented as a sequence of hex digits. 

The 32-bit words can be displayed in binary, hex and decimal. To convert a word to hex digits, each 4-bit string is converted to its hex digit equivalent.

[Reference](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).
This representation is how intermediate data is displayed. The 32-bit words are part of a chunk of a padded message

In [ ]:
#32-bits word
w = np.uint32(0xb103fe23)

print(w)               # decimal
print(f"{w:08x}")      # hex
print(f"{w:032b}")     # binary

###  Sub-section 2: sequences of 32-bit words
Sequences of 32-bits words are represented as an array. These 512-bits block is a fixed-size chunk of padded message.  16 32-bit words are broken down from the 512-bits block size (16x32). 
This is the intermediate data representation.

In [ ]:
# create numpy array with 4 32-bit unsigned integers
words = np.array([0x61626380, 0, 0xd8934, 0x00000018], dtype=np.uint32)

print(words)          
print(words.dtype)    # print data type of the array elements
print(len(words))     # print the number of elements in the array

### Sub-section 3: input messages
Input messages are data type String(str). The input messages may not have a predictable length, so it will be unsafe to use fixed-width strings. Storing all the data without truncation requires knowing the length of the longest string one would like to store in the array before the array is created. [Reference](https://numpy.org/devdocs/user/basics.strings.html)

This situations are supported using Numpy, **numpy.dtypes.StringDType**. This stores variable-width string data in UTF-8 encoding in a Numpy array(not parameterized by maximum length of an array element). Arbitrarily long or short strings can live in the same array without needing to reserve storage for padding bytes in the short strings.


In [ ]:
# represent the input message as an array of numpy data type StringDType
message = np.array(["Happy days now"], dtype=StringDType())
print(message)
print(message.dtype)  # prints the data type of input message

### Sub-section 4: 512-bit message blocks
Preprocessing/Padding involves padding a message, parsing the added message into 512-bit blocks, and setting initialization values to be used in the hash computation. This is represented as 64 bytes.

Padding is accomplished by adding a 1-bit followed by as many 0-bits as necessary to achieve the desired length. The length of the original message is also appended to the end of the padded message, expressed as a 64-bit binary number.

512-bit messages can be represented with the 8-bit unsigned integer type that uses 8 bitsof memory to store each number.



In [ ]:
# message turned into bytes.
message_bytes = message.item().encode("utf-8")

# create the 512 bit message block
message_block = np.zeros(64, dtype=np.uint8)

# put the bytes into the message block
message_block[:len(message_bytes)] = np.frombuffer(message_bytes, dtype=np.uint8)

print (message_bytes)
print(message_block)


### Sub- Section 5: Big endian Integers
Endians is the order bytes are stored in memory. Big endian stores the most significant byte(MSB) first.
Throughout this specification, the “big-endian” convention is used when expressing 32-bit words, so that within each word, the most significant bit is stored in the left-most bit position. 

In [ ]:
number = 0x1256789

# Convert the integer into a sequence of 4 bytes using big-edian byte order
# most significant byte is stored first
big_endian = number.to_bytes(4, byteorder="big")

# print bytes as a hexadecimal string
print(big_endian.hex())

## Problem 2: SHA-256 Bitwise Operations


Bitwise operations are designed to be efficient and provide strong cryptographic properties. They are combined in various ways to create more complex functions used in different stages of the hashing process for example Sigma functions, compression functions and choiceand majority functions.Bitwise operations in SHA 256 manipulate the 32-bit words, essential for the cryptographic properties of SHA 256.

Bitwise operations are divided into categories: 
1. Bitwise Logical word operations -    AND, OR, XOR, NOT
2. addition modulo  2^32           -    Addition with overflowing handling
3. Shift Operations                -    Shifts bits to the right/ left
4. Rotate Operations               -    Shifts bits to the right/left in circular fashion

Table showing summary of bitwise operations:

| Operation  | Symbol                       | Python            |What it does                              | 
|------------|------------------------------|-------------------|------------------------------------------|
|   AND      |$\land$                       |    &              |1 if both bits are 1                      |
|   OR       |$\lor$                        |   \|              |1 if at least one bit is 1                |
|   XOR      |$\oplus$                      |    ^              |1 if both bits are different              |
|   NOT      |$\neg$                        |    ~              |inverts each bit                          |
|Left Shift  |$\ll$                         |    <<             |Shifts bits  left                         |
|Right Shift |$\gg$                         |    >>             |Shifts bits right                         |
|Rotate right|$\operatorname{ROTR}^n(x)$    |Custom Function    |Rotates bits right by n positions         |
|Rotate left|$\operatorname{ROTL}^n(x)$     |Custom Function    |Rotates bits left by n Positions          |


### Sub-section 1 : Ch(x, y, z)
Choose function introduces nonlinearity into the hash function which prevents starightforwars analysis and make algorithm more resistant to attacks.
The function chooses bits from either `y` or `z`, depending on the corresponding bits in `x` i.e
- x evaluates each bit of the first word.
- if a bit in x is 1, it selects the corresponding bit from y.
- if a bit in x is 0, it selects  the corresponding bit from z.

It is defined as follows in python representation:

    Ch(x, y, z) = (x & y) ^ (~x & z)

Breakdown:
- `x & y` keeps bits from `y` wherever `x` has a 1.
- `~x & y` keeps the bits from `z` wherever `x` has 0.
-  Combine two results using XOR

In [ ]:
# Three 32-bit words
x = np.uint32(0b111100001111)
y = np.uint32(0b110011001100)
z = np.uint32(0b101010101010)

# Choose function
def Ch(x, y, z):
    """Choose: take each bit from either x or y corresponding to x"""
    return (x & y) ^ (~x & z)

# display words exactly as 32 characters in binary format
print(f"x:  {x:032b}")
print(f"y:  {y:032b}")
print(f"z:  {z:032b}")
print(f"Ch: {Ch(x, y, z):032b}")

#Test Value
assert Ch(x, y, z) == np.uint32(0b00000000000000000000110010101100)


### Sub-section 2: Maj(x, y, z)
Major function also introduces nonlinearity into the hash function which prevents starightforwars analysis and make algorithm more resistant to attacks. At each bit position, it looks at bits from  `x`, `y`, `z`. If at least two of the three bits are `1`, the output bit is `1`; otherwise it is `0`.

Defined in python as follows:

    Maj(x, y, z) = (x & y) ^ (x & z) ^ (y & z)

Breakdown:
- `x & y` compares the bits in `x` and `y`
- `x & z` compares the bits in `x` and `z`
- `y & z` compares the bits in `y` and `z`
- Combine the results using XOR

In [ ]:
# Three 32-bit words
x = np.uint32(0b1111000011110000)
y = np.uint32(0b1100110011001100)
z = np.uint32(0b1010101010101010)

# Majority Function
def Maj(x, y, z):
    #for each bit, choose majority value in x, y, z
    return (x & y) ^ (x & z) ^ (y & z)

# display words exactly as 32 characters in binary format
print(f"x:   {x:032b}")
print(f"y:   {y:032b}")
print(f"z:   {z:032b}")
print(f"Maj: {Maj(x, y, z):032b}")

#Test value
assert Maj(x, y, z) == np.uint32(0b00000000000000001110100011101000)

### Sub-section 3: Sigma0(x) — written as $\Sigma_0^{256}(x)$ in the standard
Sigma functions are used in the compression function of SHA-256. They help mix and transform the values in the state registers during each round of compression.

Defined as:

 $\Sigma_0^{256}(x)$ = ROTR²(x) ⊕ ROTR¹³(x) ⊕ ROTR²²(x)

where $\mathrm{ROTR}^{n}(x)$ is a circular right shift by n bits and `⊕` is the bitwise XOR operation. 
Each function operates on a 32-bit word represented as `x` to produce a new 32-bit word.

The bits are moved to the right but unlike the right shift, the bits that leave on the right-hand side wrap around to the left. $\Sigma_0^{256}(x)$  creates 3 rotated versions:
1. Rotate `x` right by 2 positions
2. Rotate `x` right by 13 positions
3. Rotate `x` right by 22 positions

It then combines all three results using `XOR`.

[DeepWiki](https://deepwiki.com/in3rsha/sha256-animation/3.2-sigma-functions#uppercase-sigma-functions-0-1:~:text=%CE%A30%20(usigma0)%20Function). This reference shows detailed illustrations of the usigma0 function implementation


In [ ]:
def rotr(x, n):
    """Right rotation function that shifts bits in x to the right
    by n bits and the bits wrap around to the left"""
    return (x >> n) | (x << (32 - n))

def sigma0(x):
    """Sigma function that combines all the 3 rotations of x using XOR
    operator """
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

# 32-bit word 
x = np.uint32(0b10101010101010101010101010)

# display words exactly as 32 characters in binary format
print(f"x:       {x:032b}")
print(f"ROTR 2:  {rotr(x, 2):032b}")
print(f"ROTR 13: {rotr(x, 13):032b}")
print(f"ROTR 22: {rotr(x, 22):032b}")
print(f"Sigma0:  {sigma0(x):032b}")

# Test value
assert sigma0(x) == ( rotr(x, 2) ^ rotr(x,13) ^ rotr(x,22) )

### Sub-section 4: Sigma1(x) — written as $\Sigma_1^{256}(x)$ in the standard
Sigma functions are used in the compression function of SHA-256. They help mix and transform the values in the state registers during each round of compression.

Defined as:

$\Sigma_1^{256}(x)=\mathrm{ROTR}^{6}(x)\oplus\mathrm{ROTR}^{11}(x)\oplus\mathrm{ROTR}^{25}(x)$

where $\mathrm{ROTR}^{n}(x)$ is a circular right shift by n bits and `⊕` is the bitwise XOR operation. 
Each function operates on a 32-bit word represented as `x` to produce a new 32-bit word.

The bits are moved to the right but unlike the right shift, the bits that leave on the right-hand side wrap around to the left. $\Sigma_1^{256}(x)$  creates 3 rotated versions:
1. Rotate `x` right by 6 positions
2. Rotate `x` right by 11 positions
3. Rotate `x` right by 25 positions

It then combines all three results using `XOR`.

[DeepWiki](https://deepwiki.com/in3rsha/sha256-animation/3.2-sigma-functions#uppercase-sigma-functions-0-1:~:text=%CE%A31%20(usigma1)%20Function). This reference shows detailed illustrations of the usigma1 function implementation


In [ ]:
def rotr(x, n):
    """Right rotation function that shifts bits in x to the right
    by n bits and the bits wrap around to the left"""
    return (x >> n) | (x << (32 - n))

def sigma1(x):
    """Sigma function that combines all the 3 rotations of x using XOR
    operator """
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

# 32-bit word 
x = np.uint32(0b1010101010101010101010101011)

# display words exactly as 32 characters in binary format
print(f"x:       {x:032b}")
print(f"ROTR 6:  {rotr(x, 6):032b}")
print(f"ROTR 11: {rotr(x, 11):032b}")
print(f"ROTR 25: {rotr(x, 25):032b}")
print(f"Sigma1:  {sigma1(x):032b}")

# Test value
assert sigma1(x) == ( rotr(x, 6) ^ rotr(x, 11) ^ rotr(x,25) )

### Sub-section 5: Sigma0(x) — written as $\sigma_0^{256}(x)$ in the standard
Lowercase Sigma is used in message schedule generation phase of SHA-256. It creates new words in the message schedule by transforming the existing ones (mixes bits of the earlier message words).

Defined as:

$\sigma_0(x)=\mathrm{ROTR}^{7}(x)\oplus\mathrm{ROTR}^{18}(x)\oplus\mathrm{SHR}^{3}(x)$

where $\mathrm{ROTR}^{n}(x)$ is a circular right shift by n bits. Each function operates on a 32-bit word represented as `x` to produce a new 32-bit word.

and $\mathrm{SHR}^{n}(x)$ is a right bit shift of x by `n` bits. The bits shifted are discarded and zeros enter from the left and `⊕` is the bitwise XOR operation.

Operation meaning:
1. Rotate `x` right by 7 positions
2. Rotate `x` right by 18 positions
3. Shift  `x` right by 3 positions

It then combines all three results using `XOR`.


In [ ]:
def rotr(x, n):
    """Right rotation function that shifts bits in x to the right
    by n bits and the bits wrap around to the left"""
    return (x >> n) | (x << (32 - n))

def shr(x ,n):
    """Right shift function that shifts bits in x to the right
        by n bits and the bits fall off and replaced by zeros from the left"""
    return x >> n
    
def lowerSigma0(x):
    """Sigma function that combines all the 2 rotations and 1 right shift
    of x using XOR operator """
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

# 32-bit word 
x = np.uint32(0b101010101010101010101010101110)

# display words exactly as 32 characters in binary format
print(f"x:       {x:032b}")
print(f"ROTR 7:  {rotr(x, 7):032b}")
print(f"ROTR 18: {rotr(x, 18):032b}")
print(f"SHR 3:   {shr(x, 3):032b}")
print(f"Sigma0:  {lowerSigma0(x):032b}")

# Test value
assert lowerSigma0(x) == ( rotr(x, 7) ^ rotr(x, 18) ^ shr(x,3) )

### Sub-section 6: Sigma1(x) — written as $\sigma_1^{256}(x)$ in the standard
Lowercase Sigma is used in message schedule generation phase of SHA-256. It creates new words in the message schedule by transforming the existing ones (mixes bits of the earlier message words).

Defined as:

$\sigma_0(x)=\mathrm{ROTR}^{17}(x)\oplus\mathrm{ROTR}^{19}(x)\oplus\mathrm{SHR}^{10}(x)$

where $\mathrm{ROTR}^{n}(x)$ is a circular right shift by n bits. Each function operates on a 32-bit word represented as `x` to produce a new 32-bit word.

and $\mathrm{SHR}^{n}(x)$ is a right bit shift of x by `n` bits. The bits shifted are discarded and zeros enter from the left and `⊕` is the bitwise XOR operation.

Operation meaning:
1. Rotate `x` right by 17 positions
2. Rotate `x` right by 19 positions
3. Shift  `x` right by 10 positions

It then combines all three results using `XOR`.


In [ ]:
def rotr(x, n):
    """Right rotation function that shifts bits in x to the right
    by n bits and the bits wrap around to the left"""
    return (x >> n) | (x << (32 - n))

def shr(x ,n):
    """Right shift function that shifts bits in x to the right
        by n bits and the bits fall off and replaced by zeros from the left"""
    return x >> n
    
def lowerSigma1(x):
    """Sigma function that combines all the 2 rotations and 1 right shift
    of x using XOR operator """
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

# 32-bit word 
x = np.uint32(0b10101010101110101010101010111011)

# display words exactly as 32 characters in binary format
print(f"x:       {x:032b}")
print(f"ROTR 17:  {rotr(x, 17):032b}")
print(f"ROTR 19:  {rotr(x, 19):032b}")
print(f"SHR 10:   {shr(x, 10):032b}")
print(f"Sigma1:  {lowerSigma0(x):032b}")

# Test value
assert lowerSigma1(x) == ( rotr(x, 17) ^ rotr(x, 19) ^ shr(x,10) )

## Problem 3: Generating the SHA-256 Constants


## Problem 4: Padding and Parsing Messages


### Padding the message
Padding ensures that the padded message is a multiple of 512 bits. Padding can be inserted before hash computation begins on a message before processing blocks that contain the padding.

**Suppose `l` is the bit length of the message:**

Padding is done by adding a 1-bit to end of the message.The bit is represented as 0x80 in hex or 10000000 in binary.
This is followed by a series of 0-bits (`k`) as necessary to achieve the desired length.The number of '0' bits is chosen so that the total length of the padded message is 64 bits less than a multiple of 512 bits: 448 bits.This is computed by `( (l + 1 + k ) mod 512 = 448)`
The last 64 bits store the original message.Then append the message as 64-bit big-endian integer that is equal to the number l expressed using a binary representation. [Reference](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

#### Boundary Cases:

**Scenario 1:** message is less than 448 bits
The padded message is able to fit in one 512-bits block.

**Scenario 2:** message is exactly 447 bits
The last message length fits in one block

**Scenario 3:** message is 448 bits or more
The padded message does not fit in one block so it needs more blocks. The first 512 bit block holds the 448 bit message and the next block will hold the 1 bit added to the message and zeros appended to it followed by the essage length expressed as a 64-bit-big-edian number.

**Scenario 4:** message is greater than 512 bits
Suppose the message has 600 bits, the first 512 bit block contains the first 512 bits of the message, second block has the remaining 88 bits, 1 that is added to the remaining 88 bits of the message, zeros appended and finally the message length expressed as a 64-bit-big-edian number.

### Parsing a 512 bit blocks 
After padding, the message must be split into 512 bit blocks. This is block division which ensures that the message length is a multiple of 512 bits. Blocks are represented as arrays of 64 bytes of numpy unsigned integer uint8. Each block is then processed sequentially.

#### Interpreting the message using the required byte order.
After padding and parsing the message, the 512 bit block  is then divided into 32 bit words. The 512 bit block has 16 words that are represented as unsigned integer uint32. The 64 bytes are divided into groups of 4(32 bits) to make one word. The bytes are then interpreted and ordered where the first byte is the most significant byte hence forming a 32-bit big-edian-integer.

In [ ]:
# function that accepts bytes object and yields 512 message blocks
def block_parse(msg: bytes):

    # multiply the length of the msg by 8 to get the total number of bits in the message
    msg_length = len(msg) * 8

    # 1. appending the 1 bit(represented as 0x80 or 10000000)
    padded_msg = msg + b"\x80"

    # 2. appending the required zero bits(k): msg + 1 + k = 448 k = 448 - (padded_msg), 8 bytes reserved for msg_length
    # keep adding a 0 to the msg until only 8 bytes are left(56 bytes)
    while len(padded_msg) % 64 != 56:
        padded_msg += b"\x00"  

    # 3. appending the original message length as a 64-bit integer
    padded_msg += msg_length.to_bytes(8,byteorder='big')

    # 4. Split the message in 64-byte (512-bit) chunks - Parsing the message
    blocks = []
    words = []

    for i in range(0, len(padded_msg), 64):
        # Take 64 bytes = 512 bits
        block = padded_msg[i:i + 64]
        
        # Represent the block as an array of uint8
        block = np.frombuffer(block, dtype=np.uint8)

        blocks.append(block)

        # 5. Interpret the 64 bytes as sixteen 32-bit big-endian words
        word_block = np.frombuffer(
            block.tobytes(),
            dtype=">u4"
        )

        words.append(word_block)

    return blocks, words


#600 bits representations     
# my_data = b"0xFA72B9C0E1D3F5A689BCEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF0123456789ABCDEF"
my_data = b"This sentence, including spaces and punctuation, is exactly 600 bits long."

blocks,words = block_parse(my_data)

for block in blocks:
    print(f"Block size: {len(block)} bytes ({len(block) * 8} bits)")
    #print(f"Block data: {block}\n")

    # Show each byte in hexadecimal
    hex_block = " ".join(format(b, "02x") for b in block)
    print(f"Block data (hex):\n{hex_block}\n")

    # # Convert each byte to an 8-bit binary string and join them together
    #     binary_block = "".join(format(b, '08b') for b in block)
    #     print(f"Block data (binary):\n{binary_block}\n")

# count the total number of blocks
total_blocks = len(blocks)
print(f"Total number of blocks: {total_blocks}")
print()

for word_block in words:
    print("32-bit words:")
    print([f"{word:08x}" for word in word_block])
    print()

## Problem 5: The SHA-256 Compression Function


## Problem 6: Complete SHA-256


## End